# Libraries

In [1]:
from TELF.pipeline import BlockManager
from TELF.pipeline.blocks import (
    DataBundle,
    SAVE_DIR_BUNDLE_KEY,
    SOURCE_DIR_BUNDLE_KEY,
    VultureCleanBlock,
    BeaverVocabBlock,
    BeaverDocWordBlock,
    NMFkBlock,
    HNMFkBlock,
    SemanticHNMFkBlock,
    FunctionBlock,
    ClusteringAnalyzerBlock,
    LoadDfBlock,
    LabelAnalyzerBlock
)
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.sparse as ss
import pickle
import pandas as pd
import os 

/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/pymilvus/client/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


# Load Data

In [2]:
df = pd.read_csv(Path("..") / ".." / ".." /"data" / "sample2.csv").head(50)
df.info()

EXAMPLE_OUTPUT = Path( "example_results") / 'post_process_example' 
bundle = DataBundle({'Default.df':df, 
                     SAVE_DIR_BUNDLE_KEY: EXAMPLE_OUTPUT,
                     SOURCE_DIR_BUNDLE_KEY: EXAMPLE_OUTPUT})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   eid               50 non-null     object 
 1   s2id              50 non-null     object 
 2   doi               50 non-null     object 
 3   title             50 non-null     object 
 4   abstract          50 non-null     object 
 5   year              50 non-null     int64  
 6   authors           50 non-null     object 
 7   author_ids        50 non-null     object 
 8   affiliations      50 non-null     object 
 9   funding           5 non-null      object 
 10  PACs              8 non-null      object 
 11  publication_name  50 non-null     object 
 12  subject_areas     50 non-null     object 
 13  s2_authors        50 non-null     object 
 14  s2_author_ids     50 non-null     object 
 15  citations         45 non-null     object 
 16  references        38 non-null     object 
 17 

In [3]:
vulture_block = VultureCleanBlock( init_settings={"n_jobs":1})
vocab_block = BeaverVocabBlock()
matrix_block = BeaverDocWordBlock()
factor_block = NMFkBlock(init_settings={"n_perturbs": 2, "n_iters":2})
hfactor_block = HNMFkBlock( init_settings={"nmfk_params":{"n_perturbs": 2, "n_iters":2}})

nmfk_analyzer = ClusteringAnalyzerBlock(
    tag='NMFAnalyzer',
    mode='nmf'
)
hnmfk_analyzer = ClusteringAnalyzerBlock(
    tag='HNMFAnalyzer',
    mode='hnmf'
)
extracted_cluster_only_nmfk_analyzer = ClusteringAnalyzerBlock(
    tag='ClusterOnlyAnalyzer',
    call_settings={"cluster_col": "cluster"},
    mode='label'
)
no_cluster_analyzer = ClusteringAnalyzerBlock(
    tag='NoClusterAnalyzer',
    mode=None
)

# ── after importing LabelAnalyzerBlock ──────────────────────────────────────
nmf_labels = LabelAnalyzerBlock(
    tag   = "NMFLabels",
    needs = ("NMFAnalyzer.clusters_path",)        # ← csv path from NMFAnalyzer
)

hnmfk_labels = LabelAnalyzerBlock(
    tag   = "HNMFkLabels",
    needs = ("HNMFAnalyzer.clusters_path",)       # ← list of csv paths from HNMFAnalyzer
)

cluster_only_labels = LabelAnalyzerBlock(
    tag   = "ClusterOnlyLabels",
    needs = ("ClusterOnlyAnalyzer.clusters_path",)   # ← csv path produced by label-mode run
)

no_cluster_labels = LabelAnalyzerBlock(
    tag   = "NoClusterLabels",
    needs = ("NoClusterAnalyzer.clusters_path",)  # ← csv path from pass-through analyzer
)


get_cluster_df = LoadDfBlock(
    path_extension=Path("HNMFk") / "depth_0" / 'Root',
    recursive=False,
    regex=r"cluster_for_k=.*\.csv"
)

[VultureClean] needs → (df)   provides → (df, vulture_steps)
[BeaverVocab] needs → (df)   provides → (vocabulary)
[BeaverDW] needs → (df, vocabulary)   provides → (X)
[NMFk] needs → (X)   provides → (nmfk_model, nmfk_model_path)
[HNMFk] needs → (X)   provides → (hnmfk_model, saved_path)
[NMFAnalyzer] needs → (df, nmfk_model, nmfk_model_path, vocabulary)   provides → (clusters_path)
[HNMFAnalyzer] needs → (df, hnmfk_model, vocabulary)   provides → (clusters_path)
[ClusterOnlyAnalyzer] needs → (df)   provides → (clusters_path)
[NoClusterAnalyzer] needs → (df)   provides → (clusters_path)
[NMFLabels] needs → (NMFAnalyzer.clusters_path)   provides → (result, label_paths)
[HNMFkLabels] needs → (HNMFAnalyzer.clusters_path)   provides → (result, label_paths)
[ClusterOnlyLabels] needs → (ClusterOnlyAnalyzer.clusters_path)   provides → (result, label_paths)
[NoClusterLabels] needs → (NoClusterAnalyzer.clusters_path)   provides → (result, label_paths)
[LoadDF] needs → (dir)   provides → (df, df_

In [4]:
manager = BlockManager(
    blocks = [
        vulture_block,
        vocab_block,
        matrix_block,

        factor_block,
        nmfk_analyzer,
        nmf_labels,

        hfactor_block,
        hnmfk_analyzer,
        hnmfk_labels,

        no_cluster_analyzer,
        no_cluster_labels,

        get_cluster_df,
        extracted_cluster_only_nmfk_analyzer,
        cluster_only_labels,
    ],
    databundle = bundle,
    verbose    = True,
    progress   = True,
    capture_output = "file",
)


Block (tag)                                   │ Needs (✓/✗)                                 │ Provides
──────────────────────────────────────────────────────────────────────────────────────────────────────
VultureCleanBlock (VultureClean)              │ df                                          │ ['df', 'vulture_steps']
BeaverVocabBlock (BeaverVocab)                │ df                                          │ ['vocabulary']
BeaverDocWordBlock (BeaverDW)                 │ df, vocabulary                              │ ['X']
NMFkBlock (NMFk)                              │ X                                           │ ['nmfk_model', 'nmfk_model_path']
ClusteringAnalyzerBlock (NMFAnalyzer)         │ df, nmfk_model, nmfk_model_path, vocabulary │ ['clusters_path']
LabelAnalyzerBlock (NMFLabels)                │ NMFAnalyzer.clusters_path                   │ ['result', 'label_paths']
HNMFkBlock (HNMFk)                            │ X                                           │ ['hnmfk_model

In [5]:
bundle = manager()

▶  [1/14] VultureClean …
✓  [1/14] VultureClean finished in 2.16s
▶  [2/14] BeaverVocab …
✓  [2/14] BeaverVocab finished in 0.01s
▶  [3/14] BeaverDW …
✓  [3/14] BeaverDW finished in 0.01s
▶  [4/14] NMFk …
✓  [4/14] NMFk finished in 2.51s
▶  [5/14] NMFAnalyzer …
✓  [5/14] NMFAnalyzer finished in 29.23s
▶  [6/14] NMFLabels …


SSLError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /malteos/scincl/resolve/main/tokenizer_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)')))"), '(Request ID: 686934cb-bb27-4d01-b270-95dcc54a900d)')

In [ ]:
bundle.print_tags_and_keys()

In [ ]:
bundle.NMFLabels


In [ ]:
bundle.NMFLabels.result


In [ ]:
bundle.HNMFkLabels
